# 两轴机械臂的符号建模

用一个 2R 模型查看运动学、雅可比和动力学参数，练习 MDH、标准 DH 与 PoE 的转换。按顺序运行，无需 MuJoCo，不写入文件。Franka 的坐标系检查见 [运动学 Notebook](../franka/kinematics.ipynb)。

In [ ]:
from pathlib import Path
import sys

# 根据项目标记定位根目录，支持从仓库或 notebook 目录启动。
def _project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "robot_model" / "__init__.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("找不到含 pyproject.toml 和 src/robot_model 的项目根目录")


_ROOT = _project_root(Path.cwd().resolve())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from robot_model import Robot
import sympy
from sympy import simplify, trigsimp

sympy.init_printing(use_latex=True)

## 1. 模型与惯性参数

In [ ]:
robot = Robot.from_mdh(
    "planar2r",
    [
        ("0", "l1", "0", "q"),  # alpha, a, d, theta；'q' = 关节变量
        ("0", "l2", "0", "q"),
    ],
)

spec = robot.symbols
print("convention:", robot.convention)
print("dof:", spec.dof)
spec.q, spec.dq, spec.ddq


In [ ]:
# 动力学参数符号 π（Khalil barycentric 顺序）
pi = spec.dynparms()
print("n_params =", len(pi))
pi

In [ ]:
from IPython.display import display, Markdown

s = robot.symbols
d = s.link_dyn(0)
subs = s.inertia_subs(0)

display(Markdown("### 连杆 1（索引 0）：质心处惯性 `Ie` / `I`"))
print("Ie (6 分量):", d["Ie"])
display(d["I"])

display(Markdown("### 连杆 1（索引 0）：由 `I,m,r` 得到的 `L`（平行轴）"))
display(d["L_from_I"])

display(Markdown("### 连杆 1（索引 0）：`I2L`（`I` → 用 `L` 表示）"))
for sym, expr in subs["I2L"].items():
    display(Markdown(f"${sym} = {sympy.latex(expr)}$"))

## 2. 正运动学（符号）

In [ ]:
T = robot.forward_kinematics()
T_s = trigsimp(simplify(T))
T_s

In [ ]:
# 长度取 1 m，关节角取零，检查末端位置。
lengths = {x: 1 for x in T.free_symbols if str(x) in {"l1", "l2"}}
T_zero = T.subs({**lengths, spec.q[0]: 0, spec.q[1]: 0})
assert T_zero[:3, 3] == sympy.Matrix([2, 0, 0])
T_zero


In [ ]:
dh_robot = robot.to_dh()
mdh_robot = dh_robot.to_mdh()
print("转换后的标准 DH：")
dh_robot.print_dh_table()
print("转回 MDH：")
mdh_robot.print_dh_table()


## 3. 雅可比（符号）

In [ ]:
J = robot.kinematics.J[-1]
trigsimp(simplify(J))

## 4. 逆动力学与回归矩阵

In [ ]:
tau = robot.dynamics.gen_invdyn()
print("tau shape:", tau.shape)
tau

In [ ]:
H = robot.dynamics.gen_regressor()
print("H shape:", H.shape, "(dof x n_params)")
# 只看第一行，避免刷屏
H.row(0)

## 5. 标准 DH 示例

这里单独用标准 DH 建一个两轴模型；它不是把前面的 MDH 参数原样换约定。描述同一几何时应使用 `to_dh()`，前面已演示该转换。

In [ ]:
robot_dh = Robot.from_dh(
    "planar2r_dh",
    [
        ("0", "1", "0", "q"),
        ("0", "1", "0", "q"),
    ],
)
print("convention:", robot_dh.convention)
print("空间旋量数:", len(robot_dh.to_poe().description.screws))
robot_dh.forward_kinematics().subs({robot_dh.q[0]: 0, robot_dh.q[1]: 0})